# SAM3 Model Variant Comparison

Compare **PyTorch** and **OpenVINO** SAM3 variants across two prompt modes.

**Variants:**
| Variant | Description |
|---------|-------------|
| PyTorch | Native PyTorch model (best available device: XPU > CUDA > CPU) |
| OV FP32 | OpenVINO IR, full precision |
| OV FP16 | OpenVINO IR, half precision |
| OV INT8_SYM | W8A16 weight-only compression (`compress_weights(INT8_SYM)`) |
| OV INT8_PTQ | W8A8 post-training quantization (`nncf.quantize()`) |

**Prompt modes:**
- **Text** — detect objects by category name (COCO Elephants)
- **Canvas** — stitch reference crop + target, detect by visual similarity (EDSA Vehicles)

For each mode we measure **detection count**, **latency (ms)**, and visualize mask overlays.

In [ ]:
# Copyright (C) 2025-2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

from __future__ import annotations

import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from instantlearn.data import Sample
from instantlearn.models import SAM3, SAM3OpenVINO
from instantlearn.models.sam3 import SAM3OVVariant, Sam3PromptMode
from instantlearn.models.sam3.sam3 import CanvasConfig

from instantlearn.visualizer import render_predictions

TORCH_DEVICE = "xpu" # Change to cuda for NVIDIA GPUs
OV_DEVICE = "GPU"

OV_VARIANTS = {
    "OV-FP32": SAM3OVVariant.FP32,
    "OV-FP16": SAM3OVVariant.FP16,
    "OV-INT8_SYM": SAM3OVVariant.INT8_SYM,
    "OV-INT8_PTQ": SAM3OVVariant.INT8_PTQ,
}

N_WARMUP = 2
N_MEASURE = 5

COLOR_MAP = {0: [30, 144, 255], 1: [255, 140, 0]}
CONFIDENCE = 0.3

print(f"PyTorch device: {TORCH_DEVICE}")
print(f"OpenVINO device: {OV_DEVICE}")
print(f"Variants: PyTorch + {list(OV_VARIANTS.keys())}")

In [ ]:
def measure_latency(predict_fn, target, n_warmup: int = N_WARMUP, n_measure: int = N_MEASURE) -> float:
    """Warmup + measure mean latency in ms."""
    for _ in range(n_warmup):
        predict_fn(target)
    times = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        predict_fn(target)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


def load_image_rgb(path: Path) -> np.ndarray:
    """Load image as RGB numpy array."""
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def show_comparison(
    title: str,
    image_path: Path,
    results: dict[str, dict],
    latencies: dict[str, float],
    ncols: int = 3,
) -> None:
    """Display detection results side-by-side for all variants."""
    names = list(results.keys())
    nrows = (len(names) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten()

    img_rgb = load_image_rgb(image_path)
    for i, name in enumerate(names):
        pred = results[name]
        n_det = pred["pred_masks"].shape[0] if "pred_masks" in pred else 0
        vis = render_predictions(img_rgb, pred, COLOR_MAP, show_scores=True)
        axes[i].imshow(vis)
        axes[i].set_title(f"{name}\n{n_det} det, {latencies[name]:.0f} ms", fontsize=10)
        axes[i].axis("off")
    for j in range(len(names), len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def summary_table(results_by_variant: dict[str, dict], latencies: dict[str, float]) -> None:
    """Print a compact summary table."""
    header = f"{'Variant':<14} {'Detections':>10} {'Latency (ms)':>12}"
    print(header)
    print("-" * len(header))
    for name in results_by_variant:
        n = results_by_variant[name]["pred_masks"].shape[0]
        print(f"{name:<14} {n:>10} {latencies[name]:>12.1f}")

## Dataset 1: COCO Elephants — Text Prompt Mode

Detect `"elephant"` on COCO images using text-only prompts. This tests how well
each variant handles multi-instance detection with a simple text category.

In [ ]:
COCO_DIR = Path("assets/coco/")
COCO_IMAGES = sorted(COCO_DIR.glob("*.jpg"))
COCO_CATEGORY = "elephant"

# Use first image for comparison
coco_img = COCO_IMAGES[0]

text_target = Sample(
    image_path=str(coco_img),
    categories=[COCO_CATEGORY],
    category_ids=[0],
)

# PyTorch baseline
print("Loading PyTorch model...")
pt_model = SAM3(
    device=TORCH_DEVICE,
    confidence_threshold=CONFIDENCE,
    prompt_mode=Sam3PromptMode.CLASSIC,
)
pt_model.fit(Sample(categories=[COCO_CATEGORY], category_ids=[0]))

coco_text_results: dict[str, dict] = {}
coco_text_latencies: dict[str, float] = {}

pred = pt_model.predict(text_target)
lat = measure_latency(pt_model.predict, text_target)
coco_text_results[f"PyTorch ({TORCH_DEVICE})"] = pred[0]
coco_text_latencies[f"PyTorch ({TORCH_DEVICE})"] = lat
print(f"  PyTorch ({TORCH_DEVICE}): {pred[0]['pred_masks'].shape[0]} det, {lat:.0f} ms")

del pt_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# OpenVINO variants
for name, variant in OV_VARIANTS.items():
    print(f"Loading {name}...")
    ov_model = SAM3OpenVINO(
        variant=variant,
        device=OV_DEVICE,
        confidence_threshold=CONFIDENCE,
        prompt_mode=Sam3PromptMode.CLASSIC,
    )
    ov_model.fit(Sample(categories=[COCO_CATEGORY], category_ids=[0]))

    pred = ov_model.predict(text_target)
    lat = measure_latency(ov_model.predict, text_target)
    coco_text_results[name] = pred[0]
    coco_text_latencies[name] = lat
    print(f"  {name}: {pred[0]['pred_masks'].shape[0]} det, {lat:.0f} ms")
    del ov_model

print("\nSummary")
summary_table(coco_text_results, coco_text_latencies)

In [ ]:
show_comparison(
    f'COCO Elephants — Text Prompt: "{COCO_CATEGORY}" — {coco_img.name}',
    coco_img,
    coco_text_results,
    coco_text_latencies,
)

## EDSA Vehicles — Canvas Mode

Fit on one SUV in frame 0077 with a bounding box, predict on frame 0151.
Canvas mode stitches reference crop + target into one image and uses visual
similarity together with text to guide detection.

This tests whether quantized variants lose detection capability when relying
on visual similarity transfer.

In [ ]:
EDSA_DIR = Path("assets/edsa_vehicles/")
EDSA_CATEGORY = "SUV"
canvas_config = CanvasConfig(split_ratio=0.3, crop_padding=2.0)

# Reference bbox for SUV on frame 0077
# Annotation resolution: 1920x1080, image resolution: 800x600
ann_w, ann_h = 1920, 1080
img_w, img_h = 800, 600
sx, sy = img_w / ann_w, img_h / ann_h

# SUV annotation: center_x=1176.55, center_y=495.96, w=451.33, h=551.48
cx, cy, bw, bh = 1176.55, 495.96, 451.33, 551.48
x1 = (cx - bw / 2) * sx
y1 = (cy - bh / 2) * sy
x2 = (cx + bw / 2) * sx
y2 = (cy + bh / 2) * sy

edsa_ref_path = EDSA_DIR / "EDSA-Orense-2_mp4-0077.jpg"
EDSA_REF_BBOX = np.array([[x1, y1, x2, y2]], dtype=np.float32)
edsa_canvas_target_path = EDSA_DIR / "EDSA-Orense-2_mp4-0151.jpg"

# Show reference
fig, ax = plt.subplots(figsize=(6, 4))
ref_img = load_image_rgb(edsa_ref_path)
ax.imshow(ref_img)
bx1, by1, bx2, by2 = EDSA_REF_BBOX[0].astype(int)
ax.add_patch(plt.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2, edgecolor="orange", facecolor="none"))
ax.set_title(f"Reference: {edsa_ref_path.name} — bbox for '{EDSA_CATEGORY}'")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Reference bbox (xyxy): {EDSA_REF_BBOX[0]}")

In [ ]:
edsa_ref_sample = Sample(
    image_path=str(edsa_ref_path),
    bboxes=EDSA_REF_BBOX,
    categories=[EDSA_CATEGORY],
    category_ids=[0],
)
edsa_canvas_target = Sample(image_path=str(edsa_canvas_target_path))

edsa_canvas_results: dict[str, dict] = {}
edsa_canvas_latencies: dict[str, float] = {}

# PyTorch canvas
print("Loading PyTorch model (canvas)...")
pt_model = SAM3(
    device=TORCH_DEVICE,
    confidence_threshold=CONFIDENCE,
    prompt_mode=Sam3PromptMode.CANVAS,
    canvas_config=canvas_config,
)
pt_model.fit(edsa_ref_sample)

pred = pt_model.predict(edsa_canvas_target)
lat = measure_latency(pt_model.predict, edsa_canvas_target)
edsa_canvas_results[f"PyTorch ({TORCH_DEVICE})"] = pred[0]
edsa_canvas_latencies[f"PyTorch ({TORCH_DEVICE})"] = lat
print(f"  PyTorch ({TORCH_DEVICE}): {pred[0]['pred_masks'].shape[0]} det, {lat:.0f} ms")

del pt_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# OpenVINO variants (canvas)
for name, variant in OV_VARIANTS.items():
    print(f"Loading {name} (canvas)...")
    ov_model = SAM3OpenVINO(
        variant=variant,
        device=OV_DEVICE,
        confidence_threshold=CONFIDENCE,
        prompt_mode=Sam3PromptMode.CANVAS,
        canvas_config=canvas_config,
    )
    ov_model.fit(edsa_ref_sample)

    pred = ov_model.predict(edsa_canvas_target)
    lat = measure_latency(ov_model.predict, edsa_canvas_target)
    edsa_canvas_results[name] = pred[0]
    edsa_canvas_latencies[name] = lat
    print(f"  {name}: {pred[0]['pred_masks'].shape[0]} det, {lat:.0f} ms")
    del ov_model

print("\nSummary")
summary_table(edsa_canvas_results, edsa_canvas_latencies)

In [ ]:
show_comparison(
    f"EDSA Vehicles — Canvas Mode — {edsa_canvas_target_path.name}",
    edsa_canvas_target_path,
    edsa_canvas_results,
    edsa_canvas_latencies,
)

## Summary

In [ ]:
variant_names = list(coco_text_results.keys())

print(f"{'Variant':<18} {'Text (COCO)':>20} {'Canvas (EDSA)':>20}")
print(f"{'':18} {'det':>8} {'ms':>10} {'det':>8} {'ms':>10}")
print("-" * 60)
for v in variant_names:
    n_text = coco_text_results[v]["pred_masks"].shape[0]
    l_text = coco_text_latencies[v]
    n_canvas = edsa_canvas_results[v]["pred_masks"].shape[0]
    l_canvas = edsa_canvas_latencies[v]
    print(f"{v:<18} {n_text:>8} {l_text:>10.0f} {n_canvas:>8} {l_canvas:>10.0f}")